# Use the typed Python balance client

Use `BalanceClient` to work with the background Mettler Toledo WXS205SDU simulator.
The client handles OPC UA namespace discovery, NodeIds, result types, and session
cleanup. All weight values are Python floats in grams.

| Setting | Value |
| --- | --- |
| Endpoint | `opc.tcp://127.0.0.1:4840/pylabrobot/mettler-toledo/` |
| Simulated serial number | `SIMULATED-001` |
| Initial weight | 12.345 g |

Choose this repository's `env/bin/python` as your notebook kernel. If needed,
install `PyLabRobot[opcua]` into the kernel's environment. The simulated server
must already be running. This notebook does not start or stop it.

Run the cells in order. The connection cell verifies the simulator's identity
before any measurement or control operation. The OPC UA interface has not been
verified on real hardware. See [the server guide](opcua.md) for setup and protocol
details.

For namespace discovery, browsing, and method signatures, use the
[raw OPC UA notebook](opcua-client.ipynb).

## Configure the client

The default timeout is 70 seconds per request. The wrapper discovers the namespace
automatically each time it connects.

In [ ]:
import asyncio

from asyncua import ua

from pylabrobot.mettler_toledo import BalanceClient

endpoint = "opc.tcp://127.0.0.1:4840/pylabrobot/mettler-toledo/"
balance = BalanceClient(endpoint)

## Connect and confirm the simulator

Run once per session. To reconnect later, disconnect first and run this cell again.
A refused connection usually means the background server is no longer running.

In [ ]:
await balance.connect()
try:
  info = await balance.get_info()
  if info.serial_number != "SIMULATED-001":
    raise RuntimeError(f"Expected the simulator; found {info.serial_number!r}")
except BaseException:
  await balance.disconnect()
  raise

print(f"Connected to {info.device_type} ({info.serial_number})")

## Read typed device information

`get_info()` returns a frozen `BalanceInfo` dataclass. Access properties using
normal Python attributes.

In [ ]:
info = await balance.get_info()
print(f"Model: {info.device_type}")
print(f"Firmware: {info.firmware_version}")
print(f"Capacity: {info.capacity} {info.unit}")

## Read a stable weight

A fresh simulator returns **12.345 g**. Measurements reflect any tare or zero
operations made by you or another client.

In [ ]:
weight = await balance.read_weight()
print(f"Stable weight: {weight:.6f} g")

## Read immediately

This method does not wait for stability. The simulator has a fixed load.

In [ ]:
weight = await balance.read_weight_immediately()
print(f"Immediate weight: {weight:.6f} g")

## Collect a short series

Change `sample_count` and `interval` to experiment. These are explicit measurements. The raw-client notebook also demonstrates
subscribing to the server's live `Weight` variable.

In [ ]:
sample_count = 5
interval = 0.2
samples = []

for index in range(sample_count):
  weight = await balance.read_weight_immediately()
  samples.append(weight)
  print(f"{index + 1}: {weight:.6f} g")
  if index + 1 < sample_count:
    await asyncio.sleep(interval)

## Tare the current load

Taring stores the current load as tare, making the net weight zero.

In [ ]:
await balance.tare()

## Inspect tare and net weight

On a fresh simulator, expect 12.345 g of tare and 0 g of net weight.

In [ ]:
tare = await balance.read_tare_weight()
net = await balance.read_weight()
print(f"Tare: {tare:.6f} g; net: {net:.6f} g")

## Clear tare

This clears tare without changing the zero reference.

In [ ]:
await balance.clear_tare()

## Read the restored weight

If the simulator has not been zeroed, the reading returns to 12.345 g.

In [ ]:
weight = await balance.read_weight()
print(f"After clearing tare: {weight:.6f} g")

## Handle a device error

The wrapper preserves `UaStatusCodeError` so applications can distinguish device
faults, communication errors, and timeouts. The healthy simulator succeeds here;
a failing server would enter the exception handler. Calls are never automatically
retried, since a timed-out command may still be running on the server.

In [ ]:
try:
  weight = await balance.read_weight()
  print(f"Measurement succeeded: {weight:.6f} g")
except ua.UaStatusCodeError as error:
  print(f"Balance error: {error}")
except (OSError, asyncio.TimeoutError) as error:
  print(f"Connection or timeout error: {error}")

## Optional: zero the balance

Set `run_zero` to `True` to try zeroing. It is off by default so **Run All** leaves
the initial load available for further experiments. Zero sets the current load as
the zero reference and clears tare. `clear_tare()` cannot undo it; restarting the
simulator resets its zero reference.

In [ ]:
run_zero = False

if run_zero:
  await balance.zero()
  print("Zeroed. Rerun a weight-reading cell to see the result.")
else:
  print("Zero skipped; set run_zero = True to try it.")

## Disconnect

This closes only your client session. The background server continues running.

In [ ]:
await balance.disconnect()

## Use a context manager in scripts

For short workflows, `async with` connects and disconnects automatically, including
when an exception is raised. In a notebook use top-level `await`; in a standalone
script put this inside an async function and call it with `asyncio.run()`.

In [ ]:
async with BalanceClient(endpoint) as balance:
  info = await balance.get_info()
  if info.serial_number != "SIMULATED-001":
    raise RuntimeError("Expected the simulator")
  print(f"One-shot reading: {await balance.read_weight():.6f} g")